This code is for attaching a customized excel report to a customized email, that is sent to every row of a dataframe that contains all variables needed to fill the email body and send the email. 

This notebook uses **requirements.txt** file. **requirements.txt** is a plain-text manifest that lists every Python package (and version) your project needs. Make sure you download the requirements.txt from github, and place it in your shared folder so every environment installs the exact same dependencies.


**How it connects across environments**
_This is the file that makes your folder portable:_

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; font-family: Arial, sans-serif;">
  <thead style="background-color: #f2f2f2;">
    <tr>
      <th>Environment</th>
      <th>How it uses requirements.txt</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Local</td>
      <td>pip install -r requirements.txt</td>
    </tr>
    <tr>
      <td>Databricks workspace (Repo)</td>
      <td>Add %pip install -r requirements.txt at the top of your notebook, or attach it as a cluster library</td>
    </tr>
    <tr>
      <td>Databricks Asset Bundle (dev/prod)</td>
      <td>Reference it in databricks.yml under libraries</td>
    </tr>
    <tr>
      <td>Docker</td>
      <td>COPY requirements.txt . then RUN pip install -r requirements.txt</td>
    </tr>

    <p></p>
    <table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; font-family: Arial, sans-serif;">
  <thead style="background-color: #f2f2f2;">
    <tr>
      <th>Command / Action</th>
      <th>What happens</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>pip install -r requirements.txt</td>
      <td>Installs everything listed locally</td>
    </tr>
    <tr>
      <td>%pip install -r requirements.txt</td>
      <td>Runs inside a Databricks notebook cell so the cluster gets the packages</td>
    </tr>
    <tr>
      <td>COPY + RUN pip install...</td>
      <td>Docker builds the image with identical libraries</td>
    </tr>
    <tr>
      <td>databricks.yml reference</td>
      <td>Asset Bundle installs the same list into the job cluster</td>
    </tr>
  </tbody>
</table>
  </tbody>
</table>


In [0]:
#make sure it is installed and satisfied
# requirements.txt is the contract that says “every environment must have these exact Python tools” so your notebooks run the same in local, Databricks dev, and Databricks prod. 
%pip install -r requirements.txt

#*using openpyxl (ws.append) and switching from PyDrive to the official Google client

In [0]:
# Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages

%restart_python or dbutils.library.restartPython()

Step 1 is to create a dataframe that contains all of the recipients of the emails, cc's, all variables needed for the body of the email, and field used to generate reports.  Field for generating reports could be contract numbers that are then used to query data to generate reports. 

Most likely the dataframe will be created by a query and will save into the new df.  The First Example is creating the df by adding rows. 

Fields: 
Field 1: Report Generating field
Field 2: "To" email(s)
Field 3: "CC" email(s)
... rest of variables needed

In [0]:
#Query data or create a table

#Can convert to pandas here, or later.  Email code is written using a pandas df.
#Save as email_df

#Save to volumes as email_df_[distinguishing name]

In [0]:
#Example 1: build dataframe up by adding rows of data

#table values Name, bcc email, cc email, favorite color, favorite food, score
#create temp table saved to /Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/
email_test_inbody_df = spark.createDataFrame([
    ("Gail", "gail.schneider@gsa.gov", "patrick.mckeever@gsa.gov", "green", "chocolate", 10), 
    ("Patrick", "patrick.mckeever@gsa.gov", "gail.schneider@gsa.gov", "blue", "cookies", 7),
    ("Katy", "katy.matulay@gsa.gov", "gail.schneider@gsa.gov", "blue", "ice cream", 8), 
    ("Trista", "trista.verga@gsa.gov", "gail.schneider@gsa.gov", "pink", "pizza", 8),
    ("Kim", "kimberly.tran-malan@gsa.gov", "gail.schneider@gsa.gov", "fuschia", "cake", 8),
    ("Susannah", "susannah.elizondo@gsa.gov", "gail.schneider@gsa.gov", "blue", "cookies", 7),
    ("Jamie", "jamie.kim@gsa.gov", "gail.schneider@gsa.gov", "yellow", "popsicles", 8), 
    ("Kevin", "kevin.golisano@gsa.gov", "gail.schneider@gsa.gov", "orange", "pretzels", 7), 
    ("Scott", "scott.stout@gsa.gov", "gail.schneider@gsa.gov", "purple", "cake", 8)
])

# add headers to the table
email_test_inbody_df = email_test_inbody_df.toDF("name", "email", "cc", "favorite_color", "favorite_food", "score")
display(email_test_inbody_df)

#save to /Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/
# email_test_inbody_df.write.mode("overwrite").saveAsTable("email_test_inbody_df")
test_table_path= "/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/"
email_test_inbody_df.write.mode("overwrite").option("header", "true").csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/email_test_inbody_df.csv")

    

Step 2: Create HTML body of email and save to volumes.  Include all variables needed. Matches up with email_df. 

Can create an additional cell for an html table or other item to place in the body of the email. Or this can all be in the html email body. 

In [0]:
#Html body of the email
html = """
<html>
<body>
<h1>Hi, {name}</h1>
<p>How are you?</p>
</body>
</html>
"""
#Save HTMl body as email_html_[distinguishing name]



In [0]:
#Send out the supplier email with modifications for the test
#use variables from email_test_inbody_df

email_body = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>MAS Advantage Delivery Scorecard</title>
</head>
<body style="margin:0; padding:0; background-color:#f4f4f4;">
  <table role="presentation" width="100%" cellpadding="0" cellspacing="0" border="0" style="background-color:#f4f4f4;">
    <tr>
      <td align="center" style="padding:20px 10px;">

        <table role="presentation" width="700" cellpadding="0" cellspacing="0" border="0"
               style="max-width:700px; width:100%; background-color:#ffffff; border:1px solid #dddddd;">
          <tr>
            <td style="padding:28px 32px; font-family:Arial, Helvetica, sans-serif; font-size:14px; line-height:1.5; color:#212121;">
              <h1>Hello {{name}} from Databricks!</h1>
              <h2 style="color: #ff00b5;">Sorry for the repeat email, the attachments were not sent. Hopefully, they are sent with this email.</h2>
              <p>This is another test email from Databricks to model the Supplier Report Card. Below is the email body that the contractors will receive filled in with their information. For this test a lot of the information is blank mixed in with some random personalized variables.  Attached is the excel report that the contractors will receive, but with fake orders that you received.</p>
              <p style="font-size: 12px; color: #ff00b5;"><strong>Thank you for your participation!</strong></p>

              <p style="margin:0 0 16px 0;">Dear {{name}},</p>

              <p style="margin:0 0 16px 0;">
                Our office is currently executing a reorganization and has scaled back the hands-on
                management of the Delivery Scorecard Initiative. Going forward, you will receive a
                scorecard each month (on/near the third Thursday), however, the statements have changed
                to indicate that you are passing (n/a) or failing (aka Self Improvement Required).
                No response will be required until further notice. However, it still is your
                responsibility to review the scores and associated data and make course corrections in
                your processes or catalog offerings.
                <strong>Make sure to click on ALL the tabs on the attached spreadsheet(s).</strong>
              </p>

              <p style="margin:0 0 20px 0;">
                Your CO is being copied on all messages and scorecards are taken into consideration
                when it is time for option year renewals. Please continue to take these seriously and
                work to improve every day.
              </p>

              <!-- ===== ADVANTAGE WARNING BLOCK (one of three variants) ===== -->
              <table role="presentation" width="100%" cellpadding="0" cellspacing="0" border="0"
                     style="margin:0 0 20px 0;">
                <tr>
                  <td style="background-color:{{warning_bg}}; border-left:4px solid {{warning_border}};
                             padding:12px 16px; font-family:Arial, Helvetica, sans-serif;
                             font-size:14px; line-height:1.5; color:#212121;">
                    <strong style="color:{{warning_border}};">{{warning_heading}}</strong>
                    {{warning_text}}
                  </td>
                </tr>
              </table>
              <!-- ===== END WARNING BLOCK ===== -->

              <p style="margin:0 0 16px 0;">
                Included below is a summary of your MAS Advantage monthly delivery scores. These are
                the metrics which are used to evaluate your delivery performance per contractual
                requirements.
              </p>

              <p style="margin:0 0 16px 0;">
                The attached Excel file(s) include the orders assessed to calculate your scores
                (click on ALL the tabs). The information is shared with you to verify how your scores
                were calculated.
              </p>

              <p style="margin:0 0 16px 0;">
                Please continue to submit timely order statuses via EDI or Advantage PO Portal.
              </p>

              <p style="margin:0 0 24px 0;">
                More information on this initiative can be found on
                <a href="https://www.gsa.gov" style="color:#005ea2; text-decoration:underline;">gsa.gov</a>.
              </p>

              <!-- ===== SCORECARD TABLE ===== -->
              <p style="margin:0 0 10px 0; font-size:15px; font-weight:bold;">
                {{name}} MAS Advantage {{favorite_color}} Delivery Performance:
              </p>

              <table role="presentation" width="100%" cellpadding="0" cellspacing="0" border="0"
                     style="border-collapse:collapse; margin:0 0 24px 0;
                            font-family:Arial, Helvetica, sans-serif; font-size:13px;">
                <thead>
                  <tr>
                    <th align="left" style="background-color:#eef2f5; border:1px solid #b0bec5; padding:8px 10px; color:#212121;">&nbsp;</th>
                    <th align="left" style="background-color:#eef2f5; border:1px solid #b0bec5; padding:8px 10px; color:#212121;">Minimum Acceptable Levels</th>
                    <th align="left" style="background-color:#eef2f5; border:1px solid #b0bec5; padding:8px 10px; color:#212121;">{{name}} Score</th>
                    <th align="left" style="background-color:#eef2f5; border:1px solid #b0bec5; padding:8px 10px; color:#212121;">Corrective Action Required to self improve</th>
                  </tr>
                </thead>
                <tbody>
                  <tr>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">Order status performance</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">&ge; 95%</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">{{score}}</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">{{favorite_food}}</td>
                  </tr>
                  <tr>
                    <td style="border:1px solid #b0bec5; padding:8px 10px; background-color:#fafafa;">On-Time Performance</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px; background-color:#fafafa;">&ge; 75%</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px; background-color:#fafafa;">{{score}}</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px; background-color:#fafafa;">{{favorite_color}}</td>
                  </tr>
                  <tr>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">Cancellation Performance</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">&le; 15%</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">{{score}}</td>
                    <td style="border:1px solid #b0bec5; padding:8px 10px;">{{favorite_food}}</td>
                  </tr>
                </tbody>
              </table>
              <!-- ===== END SCORECARD TABLE ===== -->

              <p style="margin:0 0 16px 0;">
                At this time, no catalogs have been suspended. We are taking a measured approach to
                administering the Delivery Scorecard initiative prior to enforcing catalog suspensions
                and/or contract terminations for consistently underperforming contracts.
              </p>

              <p style="margin:0 0 24px 0; font-weight:bold;">
                No Remediation Strategy response will be required until further notice!
              </p>

              <p style="margin:0 0 4px 0;">Thank you,</p>
              <p style="margin:0 0 24px 0;">MAS Program Management Office</p>

              <table role="presentation" width="100%" cellpadding="0" cellspacing="0" border="0">
                <tr>
                  <td style="border-top:1px solid #dddddd; padding-top:16px;
                             font-family:Arial, Helvetica, sans-serif; font-size:13px; color:#212121;">
                    <strong>Attachments:</strong>
                    <ul style="margin:8px 0 0 0; padding-left:22px;">
                      <li style="margin-bottom:4px;">GSA Advantage Metric Logic (PDF &ndash; &ldquo;How We Calculate the Metrics&rdquo; section)</li>
                      <li>Excel Attachment &ndash; Advantage</li>
                    </ul>
                  </td>
                </tr>
              </table>

            </td>
          </tr>
        </table>

      </td>
    </tr>
  </table>
</body>
</html>"""

# save this to the /tmp/ directory in Databricks
file_path = "/tmp/email_body.html"
with open(file_path, "w") as f:
  f.write(email_body)
print(f"✅ Saved template to {file_path}")

Step 3: Create a dataframe that will be queried to create the reports. 

This dataframe needs to have a matching field with the dataframe used to populate the HTML email body. And only should have the fields and rows needed for the reports. 

The first example is creating a dataframe of random values via code.  Most dataframes will most likely be made from querying the Unity Catalog, a Lakehouse connection or an imported table. 

In [0]:
#Query your data or create a table as [name]_df

#convert to pandas, because excel report code is written for a pandas df. 

#Save as [name]_df

#Save to volumes as [name]_df

In [0]:
#This example creates a pandas df of random values for testing purposes
import numpy as np
import pandas as pd

# Initial Df with names 
email_test_inbody_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/email_test_inbody_df.csv"
email_test_inbody_df = spark.read.csv(email_test_inbody_path, header=True, inferSchema=True)
#convert to pandas
email_test_inbody_df = email_test_inbody_df.toPandas()

# create a df table that contains fictitious order data with columns "name" "orderdate", "status", "deliverydate". Where the names are the names in col 1 of email_test_inbody_df:
#Status: Shipped, Backordered, Cancelled, Delivered. delvierydata has to be at least 3 days after orderdate and is only present when the status is "Delivered". All orderdate are from 8/2026.  All names need at least one order. 



rng = np.random.default_rng(42)

names = email_test_inbody_df.iloc[:, 0].dropna().astype(str).tolist()
STATUSES = ["Shipped", "Backordered", "Cancelled", "Delivered"]

records = []
for nm in names:                                    # guarantees >= 1 order per name
    for _ in range(rng.integers(1, 4)):             # 1-3 orders each
        order = pd.Timestamp("2026-08-01") + pd.Timedelta(days=int(rng.integers(0, 31)))
        status = rng.choice(STATUSES, p=[0.30, 0.20, 0.15, 0.35])
        delivery = order + pd.Timedelta(days=int(rng.integers(3, 11))) if status == "Delivered" else pd.NaT
        records.append({"name": nm, "orderdate": order, "status": status, "deliverydate": delivery})

orders_df = (pd.DataFrame(records)
             .sort_values(["name", "orderdate"], kind="stable")
             .reset_index(drop=True))
display(orders_df)

# #confirm that names in orders_df are in email_test_inbody_df
# assert set(orders_df["name"]) <= set(email_test_inbody_df["name"])
print(orders_df.shape)
print(email_test_inbody_df.shape)

for row in email_test_inbody_df.itertuples():
    email_name = row.name
    for row in orders_df.itertuples():
        orders_name = row.name
        if email_name == orders_name:
            print("name found", email_name)
    
    

#save email_test_inbody_df as pandas version
email_test_inbody_df.to_csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/email_test_inbody_pd.csv", index=False, header=True)

#save to "/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/orders_df.csv"
orders_df.to_csv("/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/orders_df.csv", index=False, header=True)   

Step 4: Create Excel reports with both standardized elements and customized filled with needed dfs.  

Loop through the searchable field from email_df to create a report for each entry and create dfs by query.  

The reports are constructed using python library openpyxl.[](https://openpyxl.readthedocs.io/en/stable/index.html). This allows you to do the same operations that you do in Excel. 

In the example 1 code, the first tab is the same for all recipients and the second one relies on querying the dataframe. 

This code saves each report as a temporary file in Databricks and then saves it to a shared g-drive folder that is shared with your service account. The excel reports can be attached with only being saved as a .tmp file, but this means they are lost when there is a restart. The additional step to save them into a g-drive folder allows for the reports to be created at a different time and protects against problem that leads to losing the .tmp files. 

If using the g-drive folder, folder can be emptied after emails are sent. See Step 6


In [0]:
#create a custom report for each email recipient. This example follows a required format for Supplier report cards.   

#make Excel report structure to match report card report
#save file to g-drive where it can be deleted from python code in another cell once the reports are sent via email

#for excel report creation
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from copy import copy

#for saving to g-drive
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError


from datetime import datetime, timedelta
#make File_NAME : [month_year] + [contract number] + "supplierReportCard.xlsx"



# Get previous month's 3-letter abbreviation + "_" + 2-digit year
today = datetime.today()
first_of_this_month = today.replace(day=1)
prev_month_date = first_of_this_month - timedelta(days=1)
month_year = prev_month_date.strftime("%b_%y")  # e.g. "May_25"

#contract_number will come from looping through df
#for now
contract_number = "cont_number"
#shared folder link:
# https://drive.google.com/drive/folders/1lxz7bEw4zWos_3faO5ZjysETQES5iJiS?usp=drive_link
# ================= CONFIGURE =================
SAVE_FOLDER_ID   = "1lxz7bEw4zWos_3faO5ZjysETQES5iJiS"      # Folder ID for upload
DELETE_FOLDER_ID = "1lxz7bEw4zWos_3faO5ZjysETQES5iJiS"    # Folder ID to clear
FILE_NAME = f'{month_year}_{contract_number}_supplierReportCard.xlsx'
# ==============================================
SCOPES = ["https://www.googleapis.com/auth/drive"]
CREDS_PATH = "/Volumes/fas_eda_analytics_prd/default/gschnvol/prj-p-ibb-dbricks-integrations-fc2403f5b873.json"



# Volume path survives restartPython(); /tmp does not
output_path = f"/Volumes/fas_eda_analytics_prd/default/gschnvol/{FILE_NAME}"
# Create Excel file
# wb = openpyxl.Workbook()
wb = Workbook()
ws = wb.active
#Populate definitions page
ws.title = "Definitions"
headers = ["Field", "Summary"]
ws.append(headers)
ws.append(["Order Status Combined Score", "The number of orders with a status entered on or before the order due date divided by the total number of orders (cancellations not included in this metric)."])
ws.append(["On Time Delivery Score", "The sum of on time orders divided by the total population of the On Time tab"])
ws.append(["Cancellation Score", "The sum of contractor and customer cancelled orders divided by the total population of the On Time and Cancellations tabs."])
ws.append(["Original Due Date", "This is original due date based on your contracted ARO Terms and Conditions and what is loaded into Advantage."])
ws.append(["Backorder Estimated Ship Date (ESD)", "This is not used for any calculation. If you backorder a line item, you must put a realistic ESD to keep custmers informed."])
ws.merge_cells("A7:A9") 
ws["A7"] = "Status"
ws["B7"] = "In Progress - You have not marked the order line with any valid status in the PO portal - This will always result in a 0 credit after the due date."
ws["B8"] = "Cancelled - Customer requested cancellation and you have not yet accepted the cancellation in the PO portal"
ws["B9"] = "Shipped - You may show something has shipped and still get a 0 in the shipstatusperfflag column if you did not mark it shipped or backordered on or beforte the due date"
ws.append(["Order Status Flag", "If you have an item with No, it counted against your score and was due to you not providing a valid status of shipped, backordered (late for any reason), or cancelled No Later Than (NLT) the original due date in PO portal."])
ws.append(["On Time Flag", "This column is used to measure your actual score (see metric logic tab). If you have an item with a 0 or blank No, it counted against your score. It measures, if the order was shipped or delivered on time AND you entered the valid status and tracking (if available) NLT the 7th day after the due date."])
ws.append(["Metric Month", "Evaluation period for all orders DUE during the metric Month"])
ws.append(["ARO Days", 'Number of days you have to either ship or deliver per your contract terms and conditions - Must advertise ARO days "Delivered" if an FOB Destination contract.  Must advertise ARO Days "Shipped" if an FOB Origin Contract'])
ws.append(["Tracking #", "If provided from a known carrier, our API uses it to capture the actual delivery date."])

#freeze headers row and make bold
ws.freeze_panes = "A1"
for cell in ws[1]: cell.font = Font(bold=True)

#Column 1 font is red #FF0000
for i in range(2, ws.max_row + 1):
    ws['A' + str(i)].font = Font(bold=True, color="FF0000")

#wrap text in 2nd column and set alignment and autofit first column
ws.column_dimensions['B'].width = 87.18
# ws["B1"].alignment = Alignment(wrap_text=True)

def wrap_column(ws, col_letter, width=40, horizontal="left", vertical="top"):
    ws.column_dimensions[col_letter].width = width          # narrow enough to force wrap
    for row in range(1, ws.max_row + 1):
        cell = ws[f"{col_letter}{row}"]
        al = copy(cell.alignment)                            # keep existing settings
        al.wrap_text = True
        al.horizontal = horizontal
        al.vertical = vertical
        cell.alignment = al
        ws.row_dimensions[row].height = None                 # let Excel auto-fit
        ws.row_dimensions[row].height = None
wrap_column(ws, "B", width=45)

#autofit for first column
max_length = 0
column_letter = "A"

for cell in ws[column_letter]:
    if cell.value:
        max_length = max(max_length, len(str(cell.value)))

ws.column_dimensions[column_letter].width = max_length + 2

#get dimensions to make borders
rowMax = ws.max_row
colMax = ws.max_column

#apply border to all cells in range
thin_border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)
for row in ws.iter_rows(min_row=1, max_row=rowMax, min_col=1, max_col=colMax):
    for cell in row:
        cell.border = thin_border


#make report sheets
ws2 = wb.create_sheet("On Time tab")

# Write it where MediaFileUpload expects it, After the upload completes, the file is temporary and cleaned up automatically
#/tmp is local to the driver node
output_path = f"/tmp/{FILE_NAME}"

# This is pure Linux local disk — no DBFS, no Spark, no UCS permissions needed
os.makedirs("/tmp", exist_ok=True)

wb.save(output_path)
print("Saved:", output_path, "Exists?", os.path.exists(output_path))
print(f"✅ Created local file: {FILE_NAME}")


print("Local xlsx exists =", os.path.exists(output_path), "size =",
      os.path.getsize(output_path) if os.path.exists(output_path) else 0)
assert os.path.exists(output_path), f"xlsx was not written: {output_path}"

# ------------------------------------------------------------------
# 2) UPLOAD
# ------------------------------------------------------------------
creds = service_account.Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
service = build("drive", "v3", credentials=creds)
print("Service account:", creds.service_account_email)

folder = service.files().get(
    fileId=SAVE_FOLDER_ID,
    fields="id, name, mimeType, driveId",
    supportsAllDrives=True,
).execute()
print("Target folder:", folder.get("name"), folder.get("id"))

media = MediaFileUpload(
    output_path,
    mimetype="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    resumable=True,
)
uploaded = service.files().create(
    body={"name": FILE_NAME, "parents": [SAVE_FOLDER_ID]},
    media_body=media,
    fields="id, name, parents, driveId",
    supportsAllDrives=True,
).execute()
print("SUCCESS:", uploaded.get("name"), "| ID:", uploaded.get("id"))



Step 5: Generate and send emails with excel reports

This code uses many callback functions inside of the iteration. This code is run for each row of the email_df. 

The first step is to retrieve the excel report that matches the current recipient.  This employs using google authentication and pointing to the same folder that the reports were saved in. 

The Second step creates the Jinja2 environment and brings in the Html email body that is saved as a temporary file. It Fills in the variables and includes any code that goes beyond filling in the variables and rendering the HTML and CSS. 

The third step fills in the email message information that comes from email_df and brings in the fully rendered email body from the second step, and then adds the report as an attachment.  

The fourth step is to send the email. 



In [0]:
#for sending email
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import pandas as pd
from email.utils import getaddresses
from jinja2 import Environment, FileSystemLoader, select_autoescape

#for retrieving report from g-drive
import os
import io
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

#for iteration
import pandas as pd
from openpyxl.utils.dataframe import dataframe_to_rows

In [0]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import pandas as pd
from email.utils import getaddresses
from jinja2 import Environment, FileSystemLoader, select_autoescape



#for retrieving report from g-drive
import os
import io
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload


#make File_NAME : [month_year] + [contract number] + "supplierReportCard.xlsx"
from datetime import datetime, timedelta

#for iteration
import pandas as pd
from openpyxl.utils.dataframe import dataframe_to_rows

#Bring in email df
# Initial Df with names take in pandas version of email_test_inbody_df 
email_test_inbody_path = "/Volumes/fas_eda_analytics_prd/default/gschnvol/Scorecards/testing/email_test_inbody_pd.csv"
email_test_inbody_df = pd.read_csv(email_test_inbody_path, header=0)


# Get previous month's 3-letter abbreviation + "_" + 2-digit year
#outside of iteration, because same for every contract number
today = datetime.today()
first_of_this_month = today.replace(day=1)
prev_month_date = first_of_this_month - timedelta(days=1)
month_year = prev_month_date.strftime("%b_%y")  # e.g. "May_25"

#functions used in larger iteration
def escape_drive_query_value(value):
    """
    Escape single quotes and backslashes for Google Drive query strings.
    """
    return value.replace("\\", "\\\\").replace("'", "\\'")

def find_file_in_drive_folder(service, folder_id, file_name):
    """
    Finds a file by exact name within a specific Google Drive folder.
    Returns the file metadata if found, otherwise None.
    """

    safe_file_name = escape_drive_query_value(file_name)

    query = (
        f"'{folder_id}' in parents "
        f"and name = '{safe_file_name}' "
        f"and trashed = false"
    )

    results = service.files().list(
        q=query,
        spaces="drive",
        fields="files(id, name, mimeType)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()

    files = results.get("files", [])

    if not files:
        return None

    # If more than one match exists, take the first.
    # You can change this behavior if needed.
    return files[0]

def download_drive_file_as_bytes(service, file_id):
    """
    Downloads a Google Drive file and returns its bytes.
    """

    request = service.files().get_media(
        fileId=file_id,
        supportsAllDrives=True
    )

    file_buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(file_buffer, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()

    file_buffer.seek(0)
    return file_buffer.read()


# Fill email body with variables from email_test_inbody_df, attach appropriate report, send emails
def make_email(row):


    # ==========================================
    # 1. Attach Supplier Report Card
    # ==========================================
    #first get attached excel report in g-drive folder
    #contract_number is in the first row of the pandas df
    #for now using name
    contract_number =  str(row.get("name", "")).strip()
    to_email = str(row.get("email", "")).strip()

    #shared folder link:
    # https://drive.google.com/drive/folders/1lxz7bEw4zWos_3faO5ZjysETQES5iJiS?usp=drive_link
     # ================= CONFIGURE =================
    SAVE_FOLDER_ID   = "1lxz7bEw4zWos_3faO5ZjysETQES5iJiS"      # Folder ID for upload
    # ==============================================
    SCOPES = ["https://www.googleapis.com/auth/drive"]
    CREDS_PATH = "/Volumes/fas_eda_analytics_prd/default/gschnvol/prj-p-ibb-dbricks-integrations-fc2403f5b873.json"
    creds = service_account.Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
    service = build("drive", "v3", credentials=creds)
    
    # Skip rows without a valid email
    if not to_email or to_email.lower() == "nan":
        return

    # Expected file name format:
    # {month_year}_{row.name}_supplierReportCard.xlsx
    FILE_NAME = f"{month_year}_{contract_number}_supplierReportCard.xlsx"

    # Search for the attachment in Google Drive
    drive_file = find_file_in_drive_folder(
        service= build("drive", "v3", credentials=creds),
        folder_id= SAVE_FOLDER_ID,
        file_name= FILE_NAME
    )

    if drive_file is None:
        print(f"No attachment found for {contract_number}: {FILE_NAME}")
        return

    # Download file bytes from Google Drive
    attachment_bytes = download_drive_file_as_bytes(
        service=service,
        file_id=drive_file["id"]
    )
    # print("SUCCESS:", uploaded.get("name"), "| ID:", uploaded.get("id"))

    #Bring in HTML email body using Jinja2
    env = Environment(
        loader=FileSystemLoader("/tmp"),
        autoescape=select_autoescape(["html"]),
    )
    template = env.get_template("email_body.html")


    # ==========================================
    # 2. Read the HTML file to use as the email body
    # ==========================================
    # Pick warning variant in Python (example)
    score = row.get("score", "")

    # Example warning logic
    try:
        score_value = float(score)
    except:
        score_value = None

    if score_value is not None and score_value >= 8:
        warning_bg = "#e6f4ea"
        warning_border = "#137333"
        warning_heading = "Passing: "
        warning_text = "Your current performance meets the minimum acceptable level."
    elif score_value is not None and score_value < 7:
        warning_bg = "#fce8e6"
        warning_border = "#c5221f"
        warning_heading = "Self Improvement Required: "
        warning_text = "Your current performance is below the minimum acceptable level."
    else:
        warning_bg = "#fff3cd"
        warning_border = "#856404"
        warning_heading = "Notice: "
        warning_text = "Please review your scorecard and associated order data."

    htmlBody = template.render(
        name=contract_number,
        score=str(row.get("score", "")).strip(),
        favorite_color=str(row.get("favorite_color", "")).strip(),
        favorite_food=str(row.get("favorite_food", "")).strip(),

        warning_bg=warning_bg,
        warning_border=warning_border,
        warning_heading=warning_heading,
        warning_text=warning_text
    )
       

    # displayHTML(htmlBody)   # preview
   
    # ==========================================
    # 3. Build message for sending
    # ==========================================
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"Databricks test email for {contract_number}"
    msg["From"] = sender_email
    msg["To"] = to_email

    # Handle CC column safely
    cc_raw = row.get("cc", "")

    if pd.notna(cc_raw) and str(cc_raw).strip() != "":
        msg["Cc"] = str(cc_raw).strip()
    else:
        msg["Cc"] = ""

   

    # Add HTML body as an "alternative" part
    body_part = MIMEMultipart("alternative")
    body_part.attach(MIMEText(htmlBody, "html"))
    msg.attach(body_part)

    # Add Excel attachment
    attachment = MIMEApplication(
        attachment_bytes,
        _subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    )

    attachment.add_header(
        "Content-Disposition",
        "attachment",
        filename=FILE_NAME
    )

    msg.attach(attachment)

    

    # recipient logging
    tos = msg.get_all('to', [])
    ccs = msg.get_all('cc', [])
    resent_tos = msg.get_all('resent-to', [])
    resent_ccs = msg.get_all('resent-cc', [])
    all_recipients = getaddresses(tos + ccs + resent_tos + resent_ccs)
    print(f"[LOG] Row: {row.get('name')} | To header: {tos} | Cc header: {ccs} | Envelope list: {all_recipients}")
    print(f"Attached file: {FILE_NAME} | Size: {len(attachment_bytes):,} bytes")



    # ==========================================
    # 4. Send the Email
    # ==========================================
    # Envelope must include To AND Cc or CC never receives
    envelope_tos = [msg["To"]]
    if msg["Cc"]:
        envelope_tos.append(msg["Cc"])

    try:
        print(f"Connecting to SMTP for {msg['To']} ...")
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(sender_email, gmail_password)
            server.sendmail(sender_email, envelope_tos, msg.as_string())
        print(f"✅ Sent! To: {msg['To']} | Cc: {msg['Cc'] or 'None'}")
    except Exception as e:
        print(f"❌ Failed for {row.get('name')}: {e}")
    
    

# ==========================================
# 4. Configure and Construct the Email
# ==========================================
sender_email = "gail.schneider@gsa.gov"


# Fetch password from Databricks secrets
gmail_password = dbutils.secrets.get(scope="gmail_scope", key="gmail_app_password")

# ==========================================
# 5. Loop
# ==========================================
for _, row in email_test_inbody_df.iterrows():
    make_email(row)

Step 5: Optional- delete all excel reports in g-drive folder upon success of sending emails

Service Account must be a manager of the shared folder to use the delete files script. 

In [0]:
#delete files in google folder 
import os
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.oauth2 import service_account



In [0]:
#delete files in google folder 
import os
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.oauth2 import service_account


#shared folder link:
# https://drive.google.com/drive/folders/1lxz7bEw4zWos_3faO5ZjysETQES5iJiS?usp=drive_link
# ================= CONFIGURE =================
SAVE_FOLDER_ID   = "1lxz7bEw4zWos_3faO5ZjysETQES5iJiS"      # Folder ID for upload
DELETE_FOLDER_ID = "1lxz7bEw4zWos_3faO5ZjysETQES5iJiS"    # Folder ID to clear
# ==============================================
SCOPES = ["https://www.googleapis.com/auth/drive"]
CREDS_PATH = "/Volumes/fas_eda_analytics_prd/default/gschnvol/prj-p-ibb-dbricks-integrations-fc2403f5b873.json"

#Delete all files in g-drive folder

# Same credentials / scopes you already use for gspread + Drive
creds = service_account.Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
service = build("drive", "v3", credentials=creds)
print("Service account:", creds.service_account_email)


def delete_all_in_folder(folder_id: str) -> None:
    query = f"'{folder_id}' in parents and trashed = false"
    page_token = None

    while True:
        try:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                pageSize=100,
            ).execute()
        except HttpError as e:
            print(f"List error: {e}")
            break

        files = response.get("files", [])
        if not files and page_token is None:
            print("Folder is already empty.")
            break

        for f in files:
            try:
                service.files().delete(
                    fileId=f["id"],
                    supportsAllDrives=True,
                ).execute()
                print(f"Deleted: {f['name']} ({f['id']})")
            except HttpError as e:
                print(f"Failed to delete {f['name']}: {e}")

        page_token = response.get("nextPageToken")
        if not page_token:
            break

if __name__ == "__main__":
    delete_all_in_folder(DELETE_FOLDER_ID)
